# 第1节：AI 安全基础与威胁建模（Threat Modeling）

本节目标：

1. 搭建 AI 安全的基本概念与术语体系（安全目标、攻击面、对手模型）
2. 认识常见攻击类型：推理阶段攻击、训练阶段攻击、服务侧攻击、供应链攻击
3. 学会用威胁建模框架把“AI 系统”拆解成组件与数据流，并识别风险点
4. 用一个小型案例（图像分类推理服务）演示：从资产/入口/能力/影响/缓解的完整分析


## 0. 环境与路径约定

如果你的 Jupyter Server 根目录设为 `notebooks/`，那么：
- `Path.cwd()` 指向 `.../项目根/notebooks`
- 项目根目录就是它的上一级

我们统一使用：

```python
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / 'data'
```

这样所有 notebook 都能稳定读取 `data/`、`models/` 等目录。

In [2]:
from pathlib import Path
import sys, platform

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR exists:", DATA_DIR.exists(), "|", DATA_DIR)
print("MODELS_DIR exists:", MODELS_DIR.exists(), "|", MODELS_DIR)


Python: 3.11.3
Platform: macOS-26.5.1-arm64-arm-64bit
CWD: /Users/jairo_wu/Documents/File/博士/课程/人工智能安全/实操
PROJECT_ROOT: /Users/jairo_wu/Documents/File/博士/课程/人工智能安全
DATA_DIR exists: False | /Users/jairo_wu/Documents/File/博士/课程/人工智能安全/data
MODELS_DIR exists: False | /Users/jairo_wu/Documents/File/博士/课程/人工智能安全/models


## 1. AI 安全到底在保护什么？（安全目标）

传统信息安全常用 CIA：
- **机密性（Confidentiality）**：不该泄露的不要泄露
- **完整性（Integrity）**：数据/模型/结果不被未授权篡改
- **可用性（Availability）**：服务可持续提供

AI 系统在 CIA 之上，还有一些更“模型相关”的目标：
- **正确性/稳健性（Correctness/Robustness）**：输入微小变化不应导致灾难性输出
- **可解释与可审计（Explainability/Auditability）**：能解释、能追责、能复现
- **公平性/合规性（Fairness/Compliance）**：满足业务与监管要求

你可以把 AI 安全理解为：
“在数据—模型—服务—反馈闭环中，保护资产、约束对手、控制风险传播”。

## 2. AI 安全术语速查（后续实验会反复用）

### 2.1 对手模型（Adversary Model）
- **能力（Capability）**：对手能做什么？能否访问模型参数/梯度？能否查询接口？能否改训练数据？
- **知识（Knowledge）**：白盒/灰盒/黑盒
- **目标（Goal）**：误分类、定向误导、降置信度、触发后门、窃取模型等
- **约束（Constraint）**：扰动幅度（如 L∞）、可见性、查询次数、预算、时延

### 2.2 攻击面（Attack Surface）
AI 系统的攻击面通常分布在：
- **数据链路**（采集/清洗/标注/训练集维护）
- **模型链路**（训练/导出/量化/部署/更新）
- **系统链路**（API 服务、鉴权、日志、监控、反馈闭环、依赖库）

### 2.3 典型攻击术语
- **Evasion Attack / 对抗样本**：推理时扰动输入使输出错误
- **Poisoning Attack / 数据投毒**：训练数据被污染，导致模型学坏
- **Backdoor / 后门**：触发器出现时输出特定目标
- **Model Extraction / 模型窃取**：通过查询接口蒸馏/拟合目标模型
- **Membership Inference / 成员推断**：判断某条数据是否参与训练
- **Prompt Injection（扩展话题）**：针对 LLM/Agent 的输入操纵


## 3. 攻击类型全景：按“系统阶段”分类

### 3.1 推理阶段（Inference-time）
- 对抗样本（FGSM/PGD/迁移攻击）
- 物理世界攻击（贴纸、屏幕反射、传感器噪声）
- 输入格式/预处理欺骗（缩放、裁剪、颜色空间、压缩）

### 3.2 训练阶段（Training-time）
- 数据投毒（label flipping、clean-label、gradient matching）
- 后门植入（trigger + target label）
- 供应链污染（训练脚本/依赖库/预训练权重）

### 3.3 服务侧（Serving/API）
- 模型窃取、成员推断、属性推断
- 资源耗尽/拒绝服务（高成本输入、并发滥用）
- 业务逻辑绕过（阈值/后处理/策略融合薄弱环节）

### 3.4 反馈闭环（Monitoring/Feedback）
- 线上反馈数据被污染，导致持续学习/再训练学坏
- 漂移未被发现，风险长期累积


## 4. 威胁建模框架：把 AI 系统拆成“资产—数据流—信任边界”

威胁建模的核心不是背框架名，而是形成一套稳定的分析动作：

1. **定义资产（Assets）**：你要保护什么？（数据、模型权重、接口、业务指标、隐私、声誉）
2. **画数据流图（DFD）**：组件、数据流、入口点、外部依赖
3. **标信任边界（Trust Boundaries）**：哪些边界外的输入都不可信？
4. **枚举威胁（Threats）**：按攻击类型/按组件/按 STRIDE/MITRE 等思路列出可能性
5. **评估风险（Risk）**：概率×影响（结合你的业务后果）
6. **设计缓解（Mitigations）**：预防/检测/响应/恢复

本课程采用一个“对 AI 更直观”的模板：

**资产 → 入口点 → 对手能力 → 攻击路径 → 影响 → 缓解措施 → 验证手段**

### 4.1 快速 DFD（文本版）

我们用一个典型的图像分类服务作为例子：

```
客户端输入图像
   │
   ▼
[API 网关/鉴权]  ----->  [日志/监控]
   │
   ▼
[预处理]  ->  [模型推理]  ->  [后处理/阈值策略]  ->  返回结果
   │                              │
   └------------->  [样本回流/反馈]┘  ->  [再训练/更新模型]
```

信任边界例子：
- API 外部输入（图片、元数据、参数）都不可信
- 回流数据不可信（可能被污染）
- 预训练权重/第三方依赖也不可信（供应链）

## 5. 典型案例解析（课程示例 1）：推理阶段对抗样本

场景：分类模型部署在线上，攻击者只能调用预测 API（黑盒），目标是让模型误判。

### 5.1 资产（Assets）
- 输出正确性（业务决策）
- 服务可用性（SLA）
- 模型本体（避免被窃取）

### 5.2 入口点（Entry Points）
- 输入图像
- 输入参数（尺寸、压缩格式、metadata）

### 5.3 对手能力（Capabilities）
- 可多次查询 API
- 可能拿到同分布数据训练替代模型（迁移攻击）

### 5.4 攻击路径（Attack Paths）
- 针对输入构造微小扰动，使输出跨过决策边界
- 利用迁移性：在替代模型上算梯度，再攻击目标模型

### 5.5 影响（Impact）
- 误分类导致业务误判（安全事故/经济损失/任务失败）
- 防御策略过强会导致误报/拒识上升（可用性损失）

### 5.6 缓解（Mitigation）
- 对抗训练、输入变换、检测/拒识、置信度校准、速率限制

### 5.7 验证（Validation）
- 用标准攻击（FGSM/PGD）评估鲁棒性
- 用迁移攻击评估黑盒风险
- 评估精度/鲁棒/时延/误报率的综合权衡

## 6. 典型案例解析（课程示例 2）：训练阶段投毒/后门

场景：训练数据来自多源采集或外部提供，攻击者能影响一部分训练样本。

### 6.1 投毒（Poisoning）
- 目标：让模型整体性能下降，或在特定子集上出错
- 入口：数据采集、标注、数据清洗脚本、数据版本管理

### 6.2 后门（Backdoor）
- 目标：在“触发器出现”时输出攻击者指定标签
- 特点：正常数据上表现可能仍很好，因此更隐蔽

### 6.3 缓解要点
- 数据治理：来源可信、版本可追溯、标注审计、异常样本检测
- 训练治理：训练过程可复现、权重来源可信、依赖库/脚本校验
- 模型检测：触发器扫描、神经元激活分析、频域/梯度特征检测（后续课会展开）

## 7. 用一个结构化表格做威胁清单（可扩展到你的平台）

下面给一个可复用的“威胁条目”数据结构：

- Threat：威胁名称
- Stage：发生阶段（数据/训练/推理/服务/闭环）
- Entry：入口点
- Capability：对手能力假设
- Impact：影响
- Mitigation：缓解措施

你可以把它输出为 CSV，后续在平台里做管理/勾选/评审。

In [3]:
import pandas as pd

threats = [
    {
        "Threat": "对抗样本误导（Evasion）",
        "Stage": "推理",
        "Entry": "输入图像",
        "Capability": "黑盒/可多次查询/可能有替代模型",
        "Impact": "误判导致业务决策错误",
        "Mitigation": "对抗训练、输入变换、检测/拒识、速率限制"
    },
    {
        "Threat": "数据投毒（Poisoning）",
        "Stage": "数据/训练",
        "Entry": "数据采集/标注/清洗脚本",
        "Capability": "可影响部分训练样本或标签",
        "Impact": "整体精度下降或特定类别失效",
        "Mitigation": "数据溯源审计、异常检测、版本管理、鲁棒训练"
    },
    {
        "Threat": "后门触发（Backdoor）",
        "Stage": "训练/推理",
        "Entry": "训练数据/预训练权重",
        "Capability": "可插入带触发器样本或篡改权重",
        "Impact": "触发器出现时定向误导输出",
        "Mitigation": "权重来源校验、后门检测、数据过滤、运行期触发器检测"
    },
    {
        "Threat": "模型窃取（Extraction）",
        "Stage": "服务",
        "Entry": "预测 API",
        "Capability": "可大量查询接口并收集输出",
        "Impact": "模型被复制、知识产权与安全能力泄露",
        "Mitigation": "限流、输出降精度、加噪、监测异常查询模式"
    },
    {
        "Threat": "成员推断（Membership Inference）",
        "Stage": "服务",
        "Entry": "预测 API",
        "Capability": "可对特定样本多次查询并分析置信度",
        "Impact": "训练数据隐私泄露",
        "Mitigation": "差分隐私训练、置信度裁剪、输出校准、隐私风险评估"
    }
]

df = pd.DataFrame(threats)
df


,Threat,Stage,Entry,Capability,Impact,Mitigation
0,对抗样本误导（Evasion）,推理,输入图像,黑盒/可多次查询/可能有替代模型,误判导致业务决策错误,对抗训练、输入变换、检测/拒识、速率限制
1,数据投毒（Poisoning）,数据/训练,数据采集/标注/清洗脚本,可影响部分训练样本或标签,整体精度下降或特定类别失效,数据溯源审计、异常检测、版本管理、鲁棒训练
2,后门触发（Backdoor）,训练/推理,训练数据/预训练权重,可插入带触发器样本或篡改权重,触发器出现时定向误导输出,权重来源校验、后门检测、数据过滤、运行期触发器检测
3,模型窃取（Extraction）,服务,预测 API,可大量查询接口并收集输出,模型被复制、知识产权与安全能力泄露,限流、输出降精度、加噪、监测异常查询模式
4,成员推断（Membership Inference）,服务,预测 API,可对特定样本多次查询并分析置信度,训练数据隐私泄露,差分隐私训练、置信度裁剪、输出校准、隐私风险评估


In [4]:
# （可选）导出威胁清单到 data/ 目录，供平台后续使用
out_path = DATA_DIR / "threat_modeling_checklist.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print("✅ exported:", out_path)


✅ exported: /Users/jairo_wu/Documents/File/博士/课程/人工智能安全/data/threat_modeling_checklist.csv


## 8. 本节练习

1. 选一个你熟悉的 AI 应用（如目标检测、语音识别、推荐系统、Agent），画出它的 DFD，并标注信任边界。
2. 按“资产→入口→能力→路径→影响→缓解→验证”写出至少 6 条威胁条目。
3. 思考：哪些威胁属于传统安全（身份鉴权/注入/供应链），哪些是 AI 特有（对抗样本/投毒/后门）？它们如何联动？

## 9. 练习


1. 选一个你熟悉的 AI 应用（如目标检测、语音识别、推荐系统、Agent），画出它的 DFD，并标注信任边界。


在人脸识别系统中，数据流从摄像头（边缘端）通过加密通道传输至中心服务器（推理引擎），并与人脸库（资产）进行比对。

信任边界 1（边缘与传输）：摄像头与网关之间，面临物理替换或中间人攻击。

信任边界 2（推理与数据库）：AI 推理服务与人脸特征向量数据库之间，这是保护核心生物信息资产的关键。

信任边界 3（管理与系统）：管理员接口与后端模型更新接口，涉及模型资产的完整性保护。

2. 按“资产→入口→能力→路径→影响→缓解→验证”写出至少 6 条威胁条目。


| **资产** | **入口** | **能力** | **路径** | **影响** | **缓解**      | **验证**   |
| ------ | ------ | ------ | ------ | ------ | ----------- | -------- |
| 模型参数   | 模型仓库   | 窃取     | 频繁请求   | 知识产权泄露 | 概率截断/API 限流 | 查询预算测试   |
| 隐私图像   | 推理接口   | 重建     | 模型反转   | 隐私泄露   | 差分隐私/脱敏     | 成员推理攻击实验 |
| 推理结果   | API    | 误导     | 对抗扰动   | 业务失控   | 对抗训练/拒识     | 对抗样本测试   |
| 训练数据   | 投毒接口   | 注入     | 数据污染   | 后门触发   | 数据清洗/审计     | 后门触发测试   |
| 计算资源   | 计算集群   | 拒绝服务   | 复杂请求   | 服务瘫痪   | 速率限制/认证     | 压力测试     |
| 模型版本   | 供应链    | 篡改     | 预训练权重  | 代码执行   | 签名验签/隔离     | 完整性扫描    |

3. 思考：哪些威胁属于传统安全（身份鉴权/注入/供应链），哪些是 AI 特有（对抗样本/投毒/后门）？它们如何联动？

威胁分类：

- **传统安全威胁**：身份鉴权缺失、数据库注入、固件供应链篡改、通信链路截获。这些威胁的本质是利用系统架构的逻辑漏洞或管理缺失。
    
- **AI 特有威胁**：对抗样本、数据投毒、模型后门、模型窃取。这些威胁利用的是模型对输入空间的非线性映射规律、训练分布的依赖性以及推理过程的统计不确定性。
    
联动机制：

在复杂攻击中，它们往往是级联的：

1. **阶段一（传统入侵）**：攻击者通过**供应链攻击（传统）**，将一个带有后门触发器（AI 特有）的模型更新包植入到门禁系统中。
    
2. **阶段二（逃逸入侵）**：正常运行时系统功能完好，但在特定攻击者出现时，攻击者佩戴特制的**对抗眼镜（AI 特有）**，触发模型后门，强制系统将特定 ID 映射为“已授权”，从而绕过**身份鉴权（传统）**。